<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/Task_3C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, max_iter=1000, tol=1e-5):
        """
        Binary Logistic Regression Classifier from scratch using Gradient Descent.
        """
        self.lr = learning_rate
        self.max_iter = max_iter
        self.tol = tol
        self.weights = None
        self.intercept = 0.0

    def _sigmoid(self, z):
        """Maps any real value into a probability range between 0 and 1."""
        #np.clip prevents mathematical overflow (RuntimeWarning) from extreme z values
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).flatten()
        n_samples, n_features = X.shape

        #Initialize weights and intercept to zero
        self.weights = np.zeros(n_features)
        self.intercept = 0.0

        #Gradient Descent optimization loop
        for iteration in range(self.max_iter):
            old_weights = self.weights.copy()
            old_intercept = self.intercept

            #Forward Pass: Compute linear combinations and apply Sigmoid
            linear_model = (X @ self.weights) + self.intercept
            y_predicted = self._sigmoid(linear_model)

            #Compute Gradients (Vectorized matrix operations)
            errors = y_predicted - y
            dw = (1 / n_samples) * (X.T @ errors)
            db = (1 / n_samples) * np.sum(errors)

            #Backward Pass: Update weights and intercept
            self.weights -= self.lr * dw
            self.intercept -= self.lr * db

            #Convergence Check: Stop early if parameter adjustments drop below tolerance
            weight_change = np.sum(np.abs(self.weights - old_weights))
            intercept_change = abs(self.intercept - old_intercept)
            if (weight_change + intercept_change) < self.tol:
                break

    def predict_proba(self, X):
        """Returns predicted probability scores for the positive class (1)."""
        X = np.asarray(X, dtype=float)
        linear_model = (X @ self.weights) + self.intercept
        return self._sigmoid(linear_model)

    def predict(self, X, threshold=0.5):
        """Returns discrete binary classifications (0 or 1) based on a threshold decision boundary."""
        probabilities = self.predict_proba(X)
        return np.where(probabilities >= threshold, 1, 0)
if __name__ == "__main__":
    #Generate 100 2D geometric samples
    np.random.seed(42)
    X_sample = np.random.randn(100, 2)

    #Classify points based on a linear threshold condition + noise
    #True relationship rule: Class 1 if (2.0 * X0 - 3.5 * X1 + 1.0) > 0
    linear_combination = 2.0 * X_sample[:, 0] - 3.5 * X_sample[:, 1] + 1.0
    y_sample = np.where(linear_combination + np.random.normal(0, 0.5, 100) > 0, 1, 0)

    #Initialize and train our model from scratch
    clf = LogisticRegressionScratch(learning_rate=0.5, max_iter=1500)
    clf.fit(X_sample, y_sample)

    print("Trained Classifier Coefficients:")
    print(f"Calculated Intercept : {clf.intercept:.4f} (Expected: Positive value)")
    print(f"Feature 0 Weight     : {clf.weights[0]:.4f} (Expected: Positive value)")
    print(f"Feature 1 Weight     : {clf.weights[1]:.4f} (Expected: Negative value)")

    #Run predictions on unseen features
    X_test = np.array([[1.5, -2.0], [-1.5, 2.0]])
    probs = clf.predict_proba(X_test)
    classes = clf.predict(X_test)

    print("\nTesting Model on Unseen Coordinate Coordinates:")
    for i, (p, c) in enumerate(zip(probs, classes)):
        print(f"Sample {i+1} Vector {X_test[i]} -> Probability of Class 1: {p:.2%} | Assigned Label: {c}")


Trained Classifier Coefficients:
Calculated Intercept : 3.0850 (Expected: Positive value)
Feature 0 Weight     : 4.9237 (Expected: Positive value)
Feature 1 Weight     : -8.5817 (Expected: Negative value)

Testing Model on Unseen Coordinate Coordinates:
Sample 1 Vector [ 1.5 -2. ] -> Probability of Class 1: 100.00% | Assigned Label: 1
Sample 2 Vector [-1.5  2. ] -> Probability of Class 1: 0.00% | Assigned Label: 0


In [2]:
import numpy as np

#TYPICAL NOTEBOOK-STYLE IMPLEMENTATION (Raw Functions, Transposed Data)
def notebook_sigmoid(z):
    #DANGEROUS: Standard notebooks do not protect against exponential overflow
    return 1 / (1 + np.exp(-z))

def notebook_fit(X, y, lr, epochs):
    #Notebooks often expect X to be transposed: (n_features, n_samples)
    X_transposed = X.T
    n_features, n_samples = X_transposed.shape

    w = np.zeros((n_features, 1))
    b = 0.0
    y = y.reshape(1, -1)

    for _ in range(epochs):
        z = np.dot(w.T, X_transposed) + b
        y_pred = notebook_sigmoid(z)

        #Matrix derivative steps
        dw = (1 / n_samples) * np.dot(X_transposed, (y_pred - y).T)
        db = (1 / n_samples) * np.sum(y_pred - y)

        w -= lr * dw
        b -= lr * db
    return w, b


#OPTIMIZED IMPLEMENTATION (Encapsulated, Standard Shapes)
class RobustLogisticRegression:
    def __init__(self, learning_rate=0.1, max_iter=1000):
        self.lr = learning_rate
        self.max_iter = max_iter
        self.weights = None
        self.intercept = 0.0

    def _sigmoid(self, z):
        #SAFE: np.clip caps values to prevent runtime exp() explosions
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.intercept = 0.0

        for _ in range(self.max_iter):
            #Safe forward pass using standard shapes: (n_samples, n_features)
            y_pred = self._sigmoid((X @ self.weights) + self.intercept)
            errors = y_pred - y

            #Vectorized gradient adjustments
            self.weights -= self.lr * ((1 / n_samples) * (X.T @ errors))
            self.intercept -= self.lr * ((1 / n_samples) * np.sum(errors))


if __name__ == "__main__":
    #Generate 150 normal data samples
    np.random.seed(42)
    X_clean = np.random.randn(150, 2)
    y_clean = np.where(X_clean[:, 0] * 2.0 - X_clean[:, 1] * 1.5 > 0, 1, 0)

    #Deliberately inject extreme outlier features to trigger an overflow/underflow bug
    X_extreme = np.vstack([X_clean, [1500.0, -3200.0], [-2500.0, 4100.0]])
    y_extreme = np.append(y_clean, [1, 0])

    print("EXECUTION SCENARIO 1: TYPICAL NOTEBOOK DESIGN:")
    print("Running notebook functions over extreme outlier data matrices...")
    try:
        #Enforce standard NumPy error warning triggers to catch hidden failure states
        with np.errstate(all='raise'):
            w_notebook, b_notebook = notebook_fit(X_extreme, y_extreme, lr=0.1, epochs=10)
            print("Notebook execution completed successfully.")
    except Exception as error_msg:
        print(f"CRASHED: Notebook code failed! Reason -> {type(error_msg).__name__}: {error_msg}")
        print("Raw formulas without input capping cause math engines to overflow.")

    print("\nEXECUTION SCENARIO 2: OUR ROBUST OBJECT-ORIENTED CLASS:")
    print("Running production-grade class over the exact same extreme outlier data matrices...")
    try:
        with np.errstate(all='raise'):
            robust_model = RobustLogisticRegression(learning_rate=0.1, max_iter=10)
            robust_model.fit(X_extreme, y_extreme)
            print("SUCCESS: Model finished training with no exceptions or mathematical errors")
            print(f"   Extracted stable weights: {robust_model.weights}")
            print(f"   Extracted stable intercept: {robust_model.intercept:.4f}")
    except Exception as error_msg:
        print(f"Unexpected Failure: {error_msg}")


EXECUTION SCENARIO 1: TYPICAL NOTEBOOK DESIGN:
Running notebook functions over extreme outlier data matrices...
CRASHED: Notebook code failed! Reason -> FloatingPointError: overflow encountered in exp
Raw formulas without input capping cause math engines to overflow.

EXECUTION SCENARIO 2: OUR ROBUST OBJECT-ORIENTED CLASS:
Running production-grade class over the exact same extreme outlier data matrices...
SUCCESS: Model finished training with no exceptions or mathematical errors
   Extracted stable weights: [ 1.46237806 -2.40362408]
   Extracted stable intercept: -0.0247


In [3]:
import numpy as np

class BinaryClassifierDiagnostics:
    @staticmethod
    def confusion_matrix(y_true, y_pred_labels):
        """
        Computes a 2x2 confusion matrix array from scratch.
        Format: [[TN, FP],
                 [FN, TP]]
        """
        y_true = np.asarray(y_true, dtype=int).flatten()
        y_pred_labels = np.asarray(y_pred_labels, dtype=int).flatten()

        tp = np.sum((y_true == 1) & (y_pred_labels == 1))
        fp = np.sum((y_true == 0) & (y_pred_labels == 1))
        tn = np.sum((y_true == 0) & (y_pred_labels == 0))
        fn = np.sum((y_true == 1) & (y_pred_labels == 0))

        return np.array([[tn, fp],
                         [fn, tp]])

    @staticmethod
    def roc_curve(y_true, y_probs):
        """
        Calculates False Positive Rates (FPR) and True Positive Rates (TPR)
        across dynamically generated probability thresholds.
        """
        y_true = np.asarray(y_true, dtype=int).flatten()
        y_probs = np.asarray(y_probs, dtype=float).flatten()

        #Sort predictions by probability in descending order
        desc_indices = np.argsort(y_probs)[::-1]
        y_true_sorted = y_true[desc_indices]
        y_probs_sorted = y_probs[desc_indices]

        #Unique thresholds to evaluate
        thresholds = np.unique(y_probs_sorted)[::-1]
        #Append boundary limits for structural completeness
        thresholds = np.append(thresholds, thresholds[-1] - 1e-5)

        fpr_list = [0.0]
        tpr_list = [0.0]

        total_positives = np.sum(y_true == 1)
        total_negatives = np.sum(y_true == 0)

        #Guard against single-class division errors
        if total_positives == 0 or total_negatives == 0:
            raise ValueError("ROC computation requires both positive and negative target classes.")

        #Scan through thresholds to compute rolling coordinate steps
        for thresh in thresholds:
            y_pred = np.where(y_probs >= thresh, 1, 0)

            tp = np.sum((y_true == 1) & (y_pred == 1))
            fp = np.sum((y_true == 0) & (y_pred == 1))

            tpr_list.append(tp / total_positives)
            fpr_list.append(fp / total_negatives)

        return np.array(fpr_list), np.array(tpr_list), thresholds

    @staticmethod
    def compute_auc(fpr, tpr):
        """
        Approximates the Area Under the Curve (AUC) using the Trapezoidal Rule.
        """
        #Ensure points are sorted by FPR to calculate accurate area increments
        sort_idx = np.argsort(fpr)
        fpr_sorted = fpr[sort_idx]
        tpr_sorted = tpr[sort_idx]

        #Trapezoid integration math: Sum of ((h_left + h_right) / 2) * width
        auc = 0.0
        for i in range(1, len(fpr_sorted)):
            width = fpr_sorted[i] - fpr_sorted[i-1]
            avg_height = (tpr_sorted[i] + tpr_sorted[i-1]) / 2.0
            auc += width * avg_height
        return auc
def display_ascii_roc(fpr, tpr, width=40, height=15):
    """Draws an inline text graph mapping the validation curve profile."""
    canvas = np.full((height, width), " ")

    #Render baseline diagonal identity curve (Random Guessing Line)
    for x in range(width):
        y = int((x / width) * height)
        canvas[min(y, height-1), x] = "."

    #Map computed validation coordinates onto the matrix plane
    for f, t in zip(fpr, tpr):
        x_coord = int(f * (width - 1))
        y_coord = int(t * (height - 1))
        canvas[y_coord, x_coord] = "█"

    print("\n   ^ [True Positive Rate / Recall]")
    for r in reversed(range(height)):
        row_str = "".join(canvas[r, :])
        print(f"1.0 | {row_str}" if r == height-1 else f"    | {row_str}")
    print("0.0 +" + "-" * width + "-> [False Positive Rate]")
    print(f"{'0.0':<7}{'1.0':>{width}}")


if __name__ == "__main__":
    #Generate mock classification validation arrays (10 elements)
    np.random.seed(1)
    y_actual    = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
    y_predicted = np.array([0.1, 0.15, 0.4, 0.35, 0.6, 0.45, 0.55, 0.8, 0.7, 0.95])

    #Assign labels using standard 0.5 threshold assignment rules
    labels = np.where(y_predicted >= 0.5, 1, 0)

    #Compute Matrix Profiling Metrics
    cm = BinaryClassifierDiagnostics.confusion_matrix(y_actual, labels)

    #Extract Curve Points and Area
    fpr, tpr, thresholds = BinaryClassifierDiagnostics.roc_curve(y_actual, y_predicted)
    auc_score = BinaryClassifierDiagnostics.compute_auc(fpr, tpr)

    #Print Diagnostics Reports
    print(f"STRUCTURAL CONFUSION MATRIX:")
    print(f"Predicted Negative | Predicted Positive")
    print(f"True Neg (TN): {cm[0,0]}   | False Pos (FP): {cm[0,1]}")
    print(f"False Neg (FN): {cm[1,0]}  | True Pos (TP): {cm[1,1]}")

    print(f"\nAREA UNDER THE CURVE:")
    print(f"Calculated AUC Metric Score: {auc_score:.4f}")

    #Display ASCII ROC curve profile
    display_ascii_roc(fpr, tpr)


STRUCTURAL CONFUSION MATRIX:
Predicted Negative | Predicted Positive
True Neg (TN): 4   | False Pos (FP): 1
False Neg (FN): 1  | True Pos (TP): 4

AREA UNDER THE CURVE:
Calculated AUC Metric Score: 0.9200

   ^ [True Positive Rate / Recall]
1.0 |        █       █       █       █      .█
    |                                    ...  
    |                                 ...     
    |        █                      ..        
    |                            ...          
    |                         ...             
    | █      █              ..                
    |                    ...                  
    |                 ...                     
    | █             ..                        
    |            ...                          
    |         ...                             
    | █     ..                                
    |    ...                                  
    | █..                                     
0.0 +-----------------------------------------> [False

In [5]:
import numpy as np

#METRICS CALCULATION CORE FROM SCRATCH
class InteractiveMetricsEngine:
    @staticmethod
    def evaluate(tp, fp, tn, fn):
        """Calculates core classification metrics directly from confusion counts."""
        #Prevent division-by-zero crashes using simple conditional checks
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0.0

        return {
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1_score
        }


if __name__ == "__main__":
    #Define our 3 real-world test cases using dictionary structures
    scenarios = {
        "Case 1: Medical Diagnosis (Rare Disease Screening)": {
            "tp": 10, "fp": 90, "tn": 890, "fn": 10,
            "desc": "Flags rare conditions. High False Positive rate."
        },
        "Case 2: Financial Fraud Detection Engine": {
            "tp": 40, "fp": 10, "tn": 430, "fn": 20,
            "desc": "Triggers on suspicious transactions. Well-balanced."
        },
        "Case 3: E-commerce Recommendation System": {
            "tp": 45, "fp": 5, "tn": 15, "fn": 35,
            "desc": "Predicts ad click-through paths. High Precision."
        }
    }

    print("=" * 68)
    print(f"{'EVALUATION PIPELINE TRACE':^68}")
    print("=" * 68)

    #Iterate through each metric scenario block
    for name, data in scenarios.items():
        metrics = InteractiveMetricsEngine.evaluate(data["tp"], data["fp"], data["tn"], data["fn"])

        print(f"\n{name}")
        print(f"   Setup: {data['desc']}")
        print(f"   onfusion Matrix -> [TP: {data['tp']}, FP: {data['fp']}, TN: {data['tn']}, FN: {data['fn']}]")
        print(f"   {'-'*62}")

        #Display each computed classification score formatted cleanly as percentages
        for metric_name, score in metrics.items():
            print(f"    {metric_name:<11} : {score:>8.2%} ({score:.4f})")

    print("\n" + "=" * 68)


                     EVALUATION PIPELINE TRACE                      

Case 1: Medical Diagnosis (Rare Disease Screening)
   Setup: Flags rare conditions. High False Positive rate.
   onfusion Matrix -> [TP: 10, FP: 90, TN: 890, FN: 10]
   --------------------------------------------------------------
    Accuracy    :   90.00% (0.9000)
    Precision   :   10.00% (0.1000)
    Recall      :   50.00% (0.5000)
    F1-Score    :   16.67% (0.1667)

Case 2: Financial Fraud Detection Engine
   Setup: Triggers on suspicious transactions. Well-balanced.
   onfusion Matrix -> [TP: 40, FP: 10, TN: 430, FN: 20]
   --------------------------------------------------------------
    Accuracy    :   94.00% (0.9400)
    Precision   :   80.00% (0.8000)
    Recall      :   66.67% (0.6667)
    F1-Score    :   72.73% (0.7273)

Case 3: E-commerce Recommendation System
   Setup: Predicts ad click-through paths. High Precision.
   onfusion Matrix -> [TP: 45, FP: 5, TN: 15, FN: 35]
   --------------------------